In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
from passwords import *

### Functions

In [2]:
# download from s3
def download_from_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

In [4]:
# create directories
def create_directories(str_dirname_output, str_dirname_model, str_variant, str_dirname_step):
    # model
    str_local_path = f'../../../{str_dirname_output}/{str_dirname_model}'
    try:
        os.mkdir(str_local_path)
    except:
        pass
    # variant
    str_local_path = f'../../../{str_dirname_output}/{str_dirname_model}/{str_variant}'
    try:
        os.mkdir(str_local_path)
    except:
        pass
    # step
    str_local_path = f'../../../{str_dirname_output}/{str_dirname_model}/{str_variant}/{str_dirname_step}'
    try:
        os.mkdir(str_local_path)
    except:
        pass

In [5]:
# download plots
def download_plots(aws_access_key_id, aws_secret_access_key, str_dirname_model, str_variant, str_dirname_step, str_project,
                   str_dirname_output):
    # init
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
    )
    # prefix
    str_prefix = f'{str_dirname_model}/02_model/{str_variant}/02_model/{str_dirname_step}/plots/'
    # list the files
    dict_response = cls_client.list_objects_v2(Bucket=str_project, Prefix=str_prefix)
    # get contents
    list_dict_contents = dict_response['Contents']
    # get the filenames
    list_str_files = [dict_contents['Key'] for dict_contents in list_dict_contents]
    # get only .png
    list_str_files = [str_file for str_file in list_str_files if '.png' in str_file]
    # download plots
    for str_bucket_path in tqdm(list_str_files):
        # get filename
        str_filename = str_bucket_path.split('/')[-1]
        str_local_path = f'../../../{str_dirname_output}/{str_dirname_model}/{str_variant}/{str_dirname_step}/{str_filename}'
        download_from_s3(
            aws_access_key_id=aws_access_key_id, 
            aws_secret_access_key=aws_secret_access_key, 
            str_local_path=str_local_path, 
            str_bucket_path=str_bucket_path, 
            str_project=str_project,
        )

### Constants

In [6]:
str_project = '20231010-gen-xii'
str_dirname_output = 'pd_plots'

### Make output directory

In [7]:
str_local_path = f'../../../{str_dirname_output}'
try:
    os.mkdir(str_local_path)
except:
    pass

### Get plots

In [8]:
%%time

# list of variants
list_str_variant = [
#     'model1',
#     'model2',
#     'model3',
#     'model4',
#     'model5',
#     'model6',
#     'noPTImodel7',
    'noPTImodel10',
]
# list of models in each variant
list_str_dirname_model = [
    '01_ad',
    '02_pricing_pd',
    '03_pricing_lgd',
]
# list of steps for each model
list_str_dirname_step = [
    '08_batch_pd_plots_valid',
    '10_batch_pd_plots_test',
]
# iterate
for str_variant in list_str_variant:
    print(f'Variant: {str_variant}')
    for str_dirname_model in list_str_dirname_model:
        print(f'Model: {str_dirname_model}')
        for str_dirname_step in list_str_dirname_step:
            print(f'Step: {str_dirname_step}')
            # make dirs
            create_directories(
                str_dirname_output=str_dirname_output, 
                str_dirname_model=str_dirname_model, 
                str_variant=str_variant, 
                str_dirname_step=str_dirname_step,
            )
            # download plots
            download_plots(
                aws_access_key_id=AWS_ACCESS_KEY_ID, 
                aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
                str_dirname_model=str_dirname_model, 
                str_variant=str_variant, 
                str_dirname_step=str_dirname_step, 
                str_project=str_project,
                str_dirname_output=str_dirname_output,
            )
    print('')

Variant: noPTImodel10
Model: 01_ad
Step: 08_batch_pd_plots_valid


100%|████████████████████████████████████████████████████████████████████████████████| 213/213 [02:22<00:00,  1.50it/s]


Step: 10_batch_pd_plots_test


100%|████████████████████████████████████████████████████████████████████████████████| 213/213 [02:22<00:00,  1.49it/s]


Model: 02_pricing_pd
Step: 08_batch_pd_plots_valid


...

Model: 03_pricing_lgd
Step: 08_batch_pd_plots_valid


100%|██████████████████████████████████████████████████████████████████████████████████| 76/76 [00:13<00:00,  5.79it/s]


Step: 10_batch_pd_plots_test


100%|██████████████████████████████████████████████████████████████████████████████████| 76/76 [00:33<00:00,  2.26it/s]


Wall time: 8min 2s
